In [1]:
# !pip install selenium beautifulsoup4 webdriver-manager

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import random
import csv
from datetime import datetime
import re

In [13]:
# detail test
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/245312233"
driver.get(detail_href)


In [28]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 한줄평
tag = soup.select_one('.desc_boxintro')
simple_desc = tag.get_text(strip=True).replace("\n", "") if tag else ""

print("한줄평은 다음과 같다: ", simple_desc)

# 태그
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

tags = extract_tags(soup)
print("태그는 다음과 같다: ", tags)

# 영업시간
# 'btn_fold2' => 눌러서 열고
# fold_detail 읽고 line_fold 하나씩 읽기
# 거기서 tit_fold 읽고 txt_detail 로 읽기 tit_fold 는 날짜만 읽기

# 더보기 버튼 누르기
def click_opening_hour_more(driver):
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_fold2"))
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1)
        return True
    except:
        return False
    
def parse_opening_hours(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    result = []

    fold_area = soup.select_one(".fold_detail")
    if not fold_area:
        return result  # 상세 페이지 없음

    lines = fold_area.select(".line_fold")

    for line in lines:
        tit = line.select_one(".tit_fold")
        date_text = tit.get_text(strip=True) if tit else ""

        date_only = re.findall(r"[월화수목금토일]", date_text)
        date_only = date_only[0] if date_only else date_text

        # 영업시간
        txt = line.select_one(".txt_detail")
        time_text = txt.get_text(" ", strip=True) if txt else ""

        result.append({
            "day": date_only,
            "hours": time_text
        })

    return result

# 더보기 버튼 클릭
click_opening_hour_more(driver)

opening_hours = parse_opening_hours(driver)
if opening_hours:
    # info['영업시간'] = opening_hours
    for oh in opening_hours:
        print(f"{oh['day']}: {oh['hours']}")
else:
    print("영업시간 상세 정보 없음")


# 가격 정보 가져오기
price_list = soup.select('.section_product .list_goods > li')
print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
prices = {}
for _, room in enumerate(price_list):
    room_type = extract_info(room, '.tit_item')
    price = extract_info(room, '.desc_item').replace("~", "")
    if room_type is None or price is None:
        continue
    prices[room_type] = price
print("가격: ", prices)
    
# 리뷰 갯수
number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
print(number_of_reviews)

# 카테고리 모텔 호텔 종류
raw_text = soup.select('.info_cate')[0]
category = raw_text.get_text(strip=True) if raw_text else None
category = category.replace("장소 카테고리", "").replace("\"", "").strip() if category else None

print("카테고리는 다음과 같다: ", category)

time.sleep(3)


한줄평은 다음과 같다:  
태그는 다음과 같다:  ['#수면실']
화: 06:00 ~ 23:00
수: 06:00 ~ 23:00
목: 06:00 ~ 23:00
금: 06:00 ~ 23:00
토: 06:00 ~ 23:00
일: 06:00 ~ 23:00
월: 06:00 ~ 23:00
총 3개의 가격 정보 수집 완료
가격:  {'평일': '13,000원', '주말/공휴일': '15,000원', '사우나 10회 쿠폰': '130,000원'}
32개
카테고리는 다음과 같다:  목욕탕,사우나


In [94]:
#  리뷰 가져오기 제일 최신 순으로 3개

# 최신순 정렬
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False
        
# 더보기 버튼 누르기
def load_reviews_with_more(driver):
    try:
        more_buttons = driver.find_elements(By.CSS_SELECTOR, ".btn_more")
        for btn in more_buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
    except:
        pass 
    

def get_recent_reviews(driver, year=2025, max_count=2):
    driver.execute_script("location.hash = '#review'")
    driver.switch_to.window(driver.window_handles[-1])
    
    print("리뷰 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(8)

    click_latest_sort(driver)
    load_reviews_with_more(driver)
    
    time.sleep(20)

    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_list = soup.select('.list_review .inner_review') # list 가져오기

    print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")

    reviews = []
    review_count = 0
    for r in review_list:
        date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
        if date is None:
            continue
        review_date = datetime.strptime(date, "%Y.%m.%d.")


        if review_date.year == year and review_count < max_count:
            describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None 
            if describe is not None:
                review_count += 1
                
                # simple cleaning
                describe = describe.replace('접기', '')
                describe = describe.replace('\n', ' ')
                describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)

                reviews.append(describe)
            else:
                continue
            
        elif review_date.year > year:
            continue
        elif review_count >= max_count:
            break
        else:
            break
    # tab close
    time.sleep(2)
    return reviews

In [92]:
# 리뷰 함수 보기
# detail test
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/1086385576#review"
driver.get(detail_href)


In [93]:
click_latest_sort(driver)
load_reviews_with_more(driver)

In [95]:
soup = BeautifulSoup(driver.page_source, "html.parser")

review_list = soup.select('.list_review .inner_review') # list 가져오기

print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")
review_count = 0
reviews = []

for r in review_list:
        date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
        if date is None:
            continue
        review_date = datetime.strptime(date, "%Y.%m.%d.")

        if review_date.year >= 2025 and review_count < 3:
            describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None  
            if describe is not None:
                review_count += 1
                
                # simple cleaning
                describe = describe.replace('접기', '')
                describe = describe.replace('\n', ' ')
                describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)
                print(describe, review_date)

            else:
                continue
        elif review_count >= 3:
            break
        else:
            break

총 20개의 리뷰 정보 수집 완료
인터넷에서 보고 패키지로 예약했는데  워터파크 이용시간 50분 중에 40분을  동력기구 2개 기다리는데 소모했어요   정작 워터파크는 거의 이용하지도 못하고  이럴거면 패키지가 왜 있는지 잘 모르겠어요  차라리 따로 구매하는게 저렴할듯   그리고 빠지 밑에는 전부 따개비가  붙어있어서 같이 간 친구 다리에서 피가 많이 났어요 대처는 따로 없음  다른 분들중에서도 긁혀서 피나거나  다치신 분들 많이 봤습니다   그리고 다른 이용객 어린여자분이  물에 빠져서 올라오기무섭다고  도와달라고 하는데도 따개비 있는걸 모르는지 밑으로 기어서 올라오라 말만하시고  엄청 겁먹으셨는데도 도와주러 올 생각도  하지 않으시더라구요   관리자 2명이 지켜만 보고 있었음  이럴거면 물놀이 하는데 안전요원 왜있나요  큰일 나면 어쩌나요..? 다른 이용객 남자분이 도와주셨어요   부산에 있어서 한번 가본건데 다신 안갈것같아요  2025-08-12 00:00:00
왠만해선 리뷰는 안남기고 싶지만 안내를 제대로 안해주고 너무 불친절합니다   2025-08-07 00:00:00
2025 해양 탐험대 참여 했는데 너무 좋았습니다. 그리고 스텝분들 너무 친절하시고 가이드도  잘해주셔서 자주 올것 갔습니다. 감사합니다.    2025-07-05 00:00:00


In [87]:
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

def extract_info(soup, tag_name):
    tag = soup.select_one(tag_name)
    text = tag.get_text(strip=True) if tag else None
    return text

# 영업 시간 더보기 버튼 누르기
def click_opening_hour_more(driver):
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_fold2"))
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1)
        return True
    except:
        return False
    
# 영업 시간
def parse_opening_hours(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    result = []

    fold_area = soup.select_one(".fold_detail")
    if not fold_area:
        return result  # 상세 페이지 없음

    lines = fold_area.select(".line_fold")

    for line in lines:
        tit = line.select_one(".tit_fold")
        date_text = tit.get_text(strip=True) if tit else ""

        date_only = re.findall(r"[월화수목금토일]", date_text)
        date_only = date_only[0] if date_only else date_text

        # 영업시간
        txt = line.select_one(".txt_detail")
        time_text = txt.get_text(" ", strip=True) if txt else ""

        result.append({
            "day": date_only,
            "hours": time_text
        })

    return result

In [88]:
# 디테일 전체 함수
def extract_details(detail_href):
    info = {}
    
    # 메인 페이지
    main_window = driver.current_window_handle
    time.sleep(5)
    
    driver.execute_script("window.open(arguments[0]);", detail_href)
    driver.switch_to.window(driver.window_handles[-1])
    
    print("상세 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(15)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # 한줄설명
    simple_desc = extract_info(soup, '.desc_boxintro')
    print("한줄설명: ", simple_desc)
    info['한줄설명'] = simple_desc
    
    # 리뷰 갯수
    number_of_reviews = extract_info(soup, '.info_num')
    print("리뷰 개수: ", number_of_reviews)
    info['리뷰 개수'] = number_of_reviews
    
    # 태그
    tags = extract_tags(soup)
    print("태그: ", tags)
    info['태그'] = tags
    
    # 영업시간
    # 더보기 버튼 클릭
    click_opening_hour_more(driver)

    opening_hours = parse_opening_hours(driver)
    if opening_hours:
        for oh in opening_hours:
            print(f"{oh['day']}: {oh['hours']}")
    else:
        print("영업시간 상세 정보 없음")
    info['영업시간'] = opening_hours # 없어도 넣고 있어도 넣음


    # 가격 정보 가져오기
    price_list = soup.select('.section_product .list_goods > li')
    print(f"총 {len(price_list)}개의 가격 정보 수집 완료")
    prices = {}
    for _, room in enumerate(price_list):
        room_type = extract_info(room, '.tit_item')
        price = extract_info(room, '.desc_item')
        if room_type is None or price is None:
            continue
        prices[room_type] = price
    print("가격: ", prices)
        
    # 리뷰
    reviews = []
    reviews = get_recent_reviews(driver)
    # 리뷰 없으면 해당 숙소 패스
    if reviews is None or len(reviews) != 1:
        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(2)
        return None
    print("리뷰: ", reviews)
    info['리뷰'] = reviews

    # 다시 검색 탭으로 전환
    driver.close()
    driver.switch_to.window(main_window)
    time.sleep(2)
    return info


In [89]:
# for testing 1 page
# 2 methods needed

def crawling(accommodation_list):
    informations = []
    for _, accommodation in enumerate(accommodation_list):
            info = {}
        # 장소명
            name = extract_info(accommodation, '.head_item > .tit_name > .link_name') 
            print("장소명:", name)
            info['장소명'] = name
        # 장소명 없으면 pass
            if name is None: 
                    continue
        
        # 평점
            score = extract_info(accommodation, '.rating > .score > em')
            print("평점:", score)
            info['평점'] = score
        # 평점 없으면 pass
            if score is None:
                    continue 
        
        # 주소
            addr = extract_info(accommodation, '.addr > p')
            print("주소:", addr)
            info['주소'] = addr
        # 주소가 없거나, 부산으로 시작하지 않으면 pass
            if addr is None or not addr.startswith("부산"):
                    continue

        # 구분
            info['구분'] = "숙박업체"
        
        # 카테고리
            category = extract_info(accommodation, '.subcategory')
            print("카테고리:", category)
            info['카테고리'] = category
        
        # 연락처
            phone_number = extract_info(accommodation, '.phone')
            print("연락처:", phone_number)
            info['연락처'] = phone_number

            a_tag = accommodation.select_one('.moreview')
            detail_href = a_tag.get('href') if a_tag else None
            details = extract_details(detail_href)
            if details is None:
                    continue
        
            info.update(details)
            info['출처'] = "카카오맵(2025)"
            print()
            print("--------------------------------------------------------------------------------------------------------")
            print("all the info: ", info)
            print("--------------------------------------------------------------------------------------------------------")
            informations.append(info)
            
    return informations

        

In [90]:
# kakao map crawling
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
url = "https://map.kakao.com/"
driver.get(url)

searchloc = '부산 수상스포츠'

search_area = driver.find_element(
    By.CSS_SELECTOR,
    "input[type='text']"
)
search_area.send_keys(searchloc)  # 검색어 전달
search_area.send_keys(Keys.ENTER)

time.sleep(5)

driver.find_element(By.XPATH, r'//*[@id="info.main.options"]/li[2]/a').send_keys(Keys.ENTER) # 장소 탭 이동

In [ ]:
# next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
# next_btn.send_keys(Keys.ENTER)

# xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[2]'
# driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)

In [91]:
import pandas as pd
import os
# 저장할 파일명
filename = "busan_activity_result.csv"

page = 1   # 현재 전체 페이지 카운트
page2 = 1  # 1~5번 버튼 중 몇 번째인지

for i in range(1, 11):
    try:
        print(f"\n=== [ {page} 페이지 수집 시작 ] ===")
        
        btn_index = page2 # 현재 그룹(1~5) 내에서의 인덱스

        xpath = f'/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a[{btn_index}]'
        
        try:
            driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)
        except:
            pass
            
        time.sleep(2) 
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        accommodation_list = soup.select('.placelist > .PlaceItem')
        print(f"찾은 업체 개수: {len(accommodation_list)}개")
        
        if len(accommodation_list) == 0:
            print("데이터가 없습니다. (혹시 로딩 에러?)")
            continue

        # 크롤링 함수 실행
        informations = crawling(accommodation_list)
        print(f"수집 완료: {len(informations)} 건")

        df = pd.DataFrame(informations)
        
        # 파일이 없으면(첫 페이지면) -> 헤더 포함해서 새로 만들기 ('w' 모드)
        if not os.path.exists(filename):
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='w')
        # 파일이 있으면(2페이지부터) -> 헤더 빼고 이어 붙이기 ('a' 모드)
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig', mode='a', header=False)
            
        print(f"-> '{filename}'에 저장 완료!")
        
        page += 1
        page2 += 1
        
        # page2가 5를 넘어가면 (즉, 6번째가 되면) -> '다음' 화살표 클릭하고 page2 초기화
        if page2 > 5:
            page2 = 1
            print(">> '다음' 버튼 클릭 (페이지 그룹 이동)")
            next_btn = driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]')
            
            # 다음 버튼이 비활성화(disabled) 상태면 종료
            if next_btn.get_attribute("disabled"):
                print("더 이상 다음 페이지가 없습니다. 종료합니다.")
                break
                
            next_btn.send_keys(Keys.ENTER)
            time.sleep(2) # 다음 묶음 로딩 대기

    except Exception as e:
        print(f"에러 발생 (중단): {e}")
        break
        
print('\n크롤링 전체 완료!')


=== [ 1 페이지 수집 시작 ] ===
찾은 업체 개수: 15개
장소명: 수영만요트경기장
평점: 4.1
주소: 부산 해운대구 해운대해변로 84
카테고리: 수상스포츠
연락처: 051-741-6440
상세 페이지로 이동 중... https://place.map.kakao.com/821880371
한줄설명:  None
리뷰 개수:  23개
태그:  ['#수상레저', '#아이와놀자', '#요트투어']
: 
총 0개의 가격 정보 수집 완료
가격:  {}
리뷰 페이지로 이동 중... https://place.map.kakao.com/821880371#review
총 20개의 리뷰 정보 수집 완료
장소명: 부산요트투어
평점: 3.9
주소: 부산 해운대구 해운대해변로 84 수영만요트경기장 1-19B
카테고리: 수상스포츠
연락처: 010-3846-6000
상세 페이지로 이동 중... https://place.map.kakao.com/499053210
한줄설명:  진짜 부산의 매력! 럭셔리한 요트에서 즐기는 새로운 체험을 경험해보시기 바랍니다.
리뷰 개수:  18개
태그:  ['#부산요트', '#부산요트대여', '#부산요트투어', '#요트투어', '#해운대요트투어', '#수상레저', '#아이와놀자']
화: 09:00 ~ 23:00
수: 09:00 ~ 23:00
목: 09:00 ~ 23:00
금: 09:00 ~ 23:00
토: 09:00 ~ 23:00
일: 09:00 ~ 23:00
월: 09:00 ~ 23:00
총 4개의 가격 정보 수집 완료
가격:  {'대인 1인 승선권 중학생 이상': '30,000원', '소인 1인 승선권 36개월~초등생': '20,000원'}
리뷰 페이지로 이동 중... https://place.map.kakao.com/499053210#review
총 18개의 리뷰 정보 수집 완료
리뷰:  ['고향친구들과  즈거운추억  만들어주신 요트 가이  분들  너무감사합니다 ']

--------------------------------------------

KeyboardInterrupt: 